# Bab 2 - Latihan Bonus: Asisten Pemeriksaan Sumur dengan Banyak Tool

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nur-Azzam-Wafiuddin/workshop/blob/uji-colab/colab/hari3/02-tools-function-calling-latihan-2.ipynb)

Ini versi **Google Colab**. Isi dan urutan contohnya sama dengan versi VS Code; yang berbeda hanya cell Persiapan di bawah, yang menyalin repository, memasang pustaka, dan mengisi API key di runtime Colab.

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kode jawaban latihan ada di bagian **Kode Jawaban Latihan** pada [PDF bab ini](https://github.com/Nur-Azzam-Wafiuddin/workshop/blob/uji-colab/hari3/02-tools-function-calling/02-tools-function-calling.pdf). Salin kodenya ke cell yang bertanda `TODO`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/Nur-Azzam-Wafiuddin/workshop.git"
REPO = "/content/workshop-genai-migas" if os.path.isdir("/content") else os.path.abspath("workshop-genai-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "uji-colab", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

Isi bagian bertanda `TODO(...)` dengan kode dari bagian **Kode Jawaban Latihan** pada PDF bab ini, lalu jalankan sel satu per satu
dengan Shift+Enter. Baris `... = None  # TODO` atau `raise NotImplementedError` berarti kode itu harus Anda ganti.

Model memilih sendiri tool yang tepat; kode menjalankannya lewat peta nama -> tool.

In [ ]:
# Langkah 1 - Menyiapkan model
import os

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

from migas_shared.field_tools import TOOL_LAPANGAN

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

## Langkah 2 - Menambah tool estimasi kehilangan produksi
Tool lapangan dari paket bersama digabung dengan satu tool baru, lalu dipetakan nama -> tool.

In [ ]:
# TODO(B2.6): buat skema KehilanganInput (minyak_bopd dan hari_henti, keduanya >= 0) dan tool estimasi_kehilangan_produksi yang mengalikan keduanya
estimasi_kehilangan_produksi = None  # TODO: ganti dengan implementasi Anda


SEMUA_TOOL = [*TOOL_LAPANGAN, estimasi_kehilangan_produksi]
PETA_TOOL = {t.name: t for t in SEMUA_TOOL}

## Langkah 3 - Loop pemanggilan tool
Loop sederhana: model meminta tool, kode menjalankan, ulangi sampai model menjawab dengan teks.

In [ ]:
def jawab(model, pertanyaan: str, maks_putaran: int = 4) -> str:
    """Loop sederhana: model meminta tool, kode menjalankan, ulangi sampai model menjawab dengan teks."""
    model_dengan_tool = model.bind_tools(SEMUA_TOOL)
    messages = [
        SystemMessage(content="Anda asisten pemeriksaan sumur di GS-SR1. Pakai tool bila perlu, jawab ringkas dalam Bahasa Indonesia."),
        HumanMessage(content=pertanyaan),
    ]
    for _ in range(maks_putaran):
        # TODO(B2.7): panggil model yang sudah diikat tool dengan daftar messages
        respons = None  # TODO: ganti dengan implementasi Anda
        if not respons.tool_calls:
            return str(respons.content)
        # TODO(B2.8): simpan permintaan tool dari model ke messages sebagai AIMessage yang membawa tool_calls
        pass  # TODO: ganti dengan implementasi Anda
        for pemanggilan in respons.tool_calls:
            try:
                # TODO(B2.9): cari tool di PETA_TOOL menurut nama pemanggilan, lalu jalankan dengan argumennya
                hasil = None  # TODO: ganti dengan implementasi Anda
            except Exception as error:  # tool yang gagal dilaporkan ke model, bukan menghentikan program
                hasil = f"Error: {error}"
            print(f"   [tool] {pemanggilan['name']}({pemanggilan['args']}) -> {hasil}")
            # TODO(B2.10): kirim hasil tool ke model sebagai ToolMessage
            pass  # TODO: ganti dengan implementasi Anda
    return "Batas putaran tercapai tanpa jawaban akhir."

## Langkah 4 - Status sumur dan water cut

In [ ]:
pertanyaan = "Bagaimana kondisi sumur SR-020, dan berapa water cut-nya?"
print(f"Pengguna: {pertanyaan}")
print("Asisten :", jawab(model, pertanyaan), "\n")

## Langkah 5 - Estimasi kehilangan produksi

In [ ]:
pertanyaan = "Sumur KN-033 menunggu pulling selama 12 hari. Jika produksi normalnya 45 bopd, berapa produksi yang hilang?"
print(f"Pengguna: {pertanyaan}")
print("Asisten :", jawab(model, pertanyaan), "\n")

## Langkah 6 - Konversi tekanan

In [ ]:
pertanyaan = "Tekanan kepala sumur ML-012 tercatat 900 psi. Berapa bar?"
print(f"Pengguna: {pertanyaan}")
print("Asisten :", jawab(model, pertanyaan), "\n")